# AnyTimeBert — Train ElasticBERT on 5 GLUE tasks

Base: `OpenMOSS-Team/elasticbert-base` (multi-exit pretrained)

Flow: **Setup → HF login → Prepare data → Train (push to HF) → Verify**

## 1. Setup

In [ ]:
# Mount Drive (Colab only) — skip if local
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/spd'
    !git clone https://github.com/wicaksonolxn/spd.git $REPO_PATH 2>/dev/null || (cd $REPO_PATH && git pull)
except Exception:
    REPO_PATH = '..'   # local: notebook is in scripts/, repo is parent

%cd $REPO_PATH

In [ ]:
!pip install -q -r AnyTimeBert/requirements.txt
!pip install -q datasets huggingface_hub psutil pynvml

## 2. HF login (Colab Secrets)

Set `HF_TOKEN` in Colab Secrets (key icon, sidebar). Local: `.env` already loaded by `config.py`.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HF_USER']  = userdata.get('HF_USER', 'wicaksonolxn')
except Exception:
    pass   # local: .env already loaded by shared.load_env()

from huggingface_hub import login, whoami
if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('Logged in as:', whoami().get('name', '?'))

## 3. Prepare data

Pulls GLUE from HF `datasets`, dumps TSVs in format ElasticBERT reference expects. One-time per task.

In [ ]:
import sys
sys.path.insert(0, '.')
from AnyTimeBert.prepare_data import prepare_all

prepare_all()                         # all 5 tasks: SST-2, MRPC, QNLI, RTE, CoLA
# prepare_all(only=['RTE'])           # one task only

## 4. Train

Each task auto-pushes checkpoint to `HF_USER/elasticbert-base-{task}-ee` on success.

In [ ]:
from AnyTimeBert.train import train, train_all

# === Pick mode ===
# Smoke test (smallest task, ~5 min on 8GB):
ckpt = train('RTE')

# All 5 tasks (~1.5-3 hr on 8GB GPU, fp16):
# train_all()

# Custom hyperparams:
# train('SST-2', epochs=3, batch_size=8, lr=3e-5)

## 5. Verify HF push

In [ ]:
from huggingface_hub import list_repo_files
from AnyTimeBert.config import hf_repo_for, TASKS

for t in TASKS:
    repo = hf_repo_for(t)
    try:
        files = list_repo_files(repo)
        print(f'  {repo}: {len(files)} files')
    except Exception as e:
        print(f'  {repo}: not pushed yet ({e})')